# Analyze Missing Soil Cells

This notebook identifies cells that need soil repair/download support for the `data_sat_soil_10d` pipeline.

It separates two categories:

```text
category1_with_wheat_missing_soil:
  A static wheat/soil-cell row exists and has positive in-county wheat,
  but soil feature values are missing.

category2_sat_only_no_soil_row:
  A fused satellite cell exists, but no matching static wheat/soil-cell row
  exists for the same county-year after rounded lon/lat matching.
```

The notebook writes occurrence-level CSVs and a unique download-point CSV. The unique file aggregates over years and county-years, so the same rounded lon/lat is listed once even if it appears repeatedly.

## Requirements

Use an environment with:

```text
pandas
numpy
```

The notebook assumes it is run from the project root `Code_minimum`.

In [1]:
from __future__ import annotations

from pathlib import Path
from datetime import date, timedelta

import numpy as np
import pandas as pd

## Configuration

By default this uses the same 10d county-year whitelist as the integration dataset. If a county-year is not in `Data_Windowed_{year}_10d`, it is not analyzed here.

In [12]:
BASE_DIR = Path.cwd()

YEARS = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2025]
JOIN_COORD_DECIMALS = 4

WINDOW_START_DM = "7.4"
WINDOW_END_DM = "31.8"
WINDOW_FREQUENCY_DAYS = 10

SOIL_DIR = BASE_DIR / "Data_Astatic_wheat_soil_cells"

# Category 1 threshold: the cell has wheat support inside this county.
POSITIVE_INCOUNTY_WHEAT_THRESHOLD = 1

# Usually the problematic soil rows have all soil means missing. Keeping both flags is useful.
CATEGORY1_USE_ALL_SOIL_MISSING = True

OUTPUT_CATEGORY1_OCCURRENCES = BASE_DIR / "missing_soil_category1_with_wheat_occurrences.csv"
OUTPUT_CATEGORY2_OCCURRENCES = BASE_DIR / "missing_soil_category2_sat_only_occurrences.csv"
OUTPUT_UNIQUE_DOWNLOAD_POINTS = BASE_DIR / "missing_soil_unique_download_points.csv"
OUTPUT_UNIQUE_DOWNLOAD_LONLAT = BASE_DIR / "missing_soil_unique_download_lonlat.csv"
OUTPUT_COUNTY_YEAR_SUMMARY = BASE_DIR / "missing_soil_county_year_summary.csv"

## Helpers

`discover_county_years` uses `Data_Windowed_{year}_10d` as the inclusion whitelist. Satellite observations are optionally filtered to the same season window before looking for satellite-only cells.

In [3]:
def parse_day_month(value: object, dummy_year: int = 2000) -> date:
    day_text, month_text = str(value).split(".", 1)
    return date(dummy_year, int(month_text), int(day_text))


def make_windows(start_dm: str, end_dm: str, frequency_days: int) -> list[tuple[date, date]]:
    start = parse_day_month(start_dm)
    end = parse_day_month(end_dm)
    windows = []
    current = start
    while current + timedelta(days=frequency_days - 1) <= end:
        window_end = current + timedelta(days=frequency_days - 1)
        windows.append((current, window_end))
        current = window_end + timedelta(days=1)
    return windows


WINDOWS = make_windows(WINDOW_START_DM, WINDOW_END_DM, WINDOW_FREQUENCY_DAYS)


def in_10d_season_mask(date_series: pd.Series) -> pd.Series:
    parsed = date_series.map(parse_day_month)
    mask = pd.Series(False, index=date_series.index)
    for start, end in WINDOWS:
        mask |= (parsed >= start) & (parsed <= end)
    return mask


def county_id_from_parts(state_id: str | int, county_num: str | int) -> str:
    return f"{int(state_id)}_{int(county_num)}"


def discover_county_years() -> list[dict]:
    records = []
    for year in YEARS:
        windowed_dir = BASE_DIR / f"Data_Windowed_{year}_10d"
        if not windowed_dir.exists():
            continue

        for windowed_path in sorted(windowed_dir.glob(f"Harvester_{year}_*_*.csv")):
            parts = windowed_path.stem.split("_")
            if len(parts) != 4:
                continue
            state_id = int(parts[2])
            county_num = int(parts[3])
            county_id = county_id_from_parts(state_id, county_num)
            records.append(
                {
                    "year": year,
                    "state_id": state_id,
                    "county_num": county_num,
                    "county_id": county_id,
                    "windowed_path": windowed_path,
                    "fused_path": BASE_DIR / f"Data_Fused_{year}" / f"Harvester_{year}_{state_id}_{county_num}.csv",
                    "soil_path": SOIL_DIR / f"county_{county_id}_{year}_static_wheat_soil_cells.csv",
                }
            )
    return records


def add_join_coords(df: pd.DataFrame, lon_col: str = "longitude", lat_col: str = "latitude") -> pd.DataFrame:
    out = df.copy()
    out["longitude_debug"] = out[lon_col].round(JOIN_COORD_DECIMALS)
    out["latitude_debug"] = out[lat_col].round(JOIN_COORD_DECIMALS)
    return out

## Discover Soil Feature Columns

The missing-soil test uses columns ending in `_mean_1km`. Count columns are useful for source debugging but not for deciding if the soil feature vector exists.

In [4]:
COUNTY_YEARS = discover_county_years()
print(f"Whitelisted county-years: {len(COUNTY_YEARS):,}")

first_soil_path = next(r["soil_path"] for r in COUNTY_YEARS if r["soil_path"].exists())
first_cols = pd.read_csv(first_soil_path, nrows=0).columns.tolist()
SOIL_MEAN_COLS = [c for c in first_cols if c.endswith("_mean_1km")]
SOIL_COUNT_COLS = [c for c in first_cols if c.endswith("_soil_pixel_count_1km")]

print(f"Soil mean columns: {len(SOIL_MEAN_COLS)}")
print(f"Soil count columns: {len(SOIL_COUNT_COLS)}")
SOIL_MEAN_COLS

Whitelisted county-years: 1,872
Soil mean columns: 18
Soil count columns: 18


['ocd_0-5cm_mean_mean_1km',
 'soc_0-5cm_mean_mean_1km',
 'nitrogen_0-5cm_mean_mean_1km',
 'phh2o_gradient_30_100_minus_0_30_mean_1km',
 'clay_gradient_30_100_minus_0_30_mean_1km',
 'sand_gradient_30_100_minus_0_30_mean_1km',
 'ocd_0_30_mean_1km',
 'awc_0_30_mean_1km',
 'nitrogen_gradient_30_100_minus_0_30_mean_1km',
 'bdod_0-5cm_mean_mean_1km',
 'soc_gradient_30_100_minus_0_30_mean_1km',
 'soc_0_30_mean_1km',
 'ocd_30-60cm_mean_mean_1km',
 'phh2o_30-60cm_mean_mean_1km',
 'wv0010_0-5cm_mean_mean_1km',
 'nitrogen_0_30_mean_1km',
 'nitrogen_15-30cm_mean_mean_1km',
 'phh2o_30_100_mean_1km']

## Analyze One County-Year

Category 1 comes from the static soil-cell file. Category 2 comes from fused satellite coordinates that do not match any static soil-cell coordinate for that county-year.

In [5]:
def analyze_one_county_year(record: dict) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    year = record["year"]
    county_id = record["county_id"]
    soil_path = record["soil_path"]
    fused_path = record["fused_path"]

    summary = {
        "year": year,
        "county_id": county_id,
        "soil_exists": soil_path.exists(),
        "fused_exists": fused_path.exists(),
        "status": "started",
    }

    cat1 = pd.DataFrame()
    cat2 = pd.DataFrame()

    if not soil_path.exists() or not fused_path.exists():
        summary["status"] = "missing_source"
        return cat1, cat2, summary

    try:
        soil = pd.read_csv(soil_path)
    except pd.errors.EmptyDataError:
        soil = pd.DataFrame()

    if soil.empty:
        summary.update({"status": "empty_soil", "soil_cells": 0})
        return cat1, cat2, summary

    soil = add_join_coords(soil)
    soil_keys = soil[["longitude_debug", "latitude_debug"]].drop_duplicates()

    local_soil_cols = [c for c in SOIL_MEAN_COLS if c in soil.columns]
    missing_any_soil = soil[local_soil_cols].isna().any(axis=1)
    missing_all_soil = soil[local_soil_cols].isna().all(axis=1)
    positive_incounty = soil["static_wheat_pixels_incounty"].fillna(0) >= POSITIVE_INCOUNTY_WHEAT_THRESHOLD

    cat1_mask = positive_incounty & (missing_all_soil if CATEGORY1_USE_ALL_SOIL_MISSING else missing_any_soil)
    if cat1_mask.any():
        keep_cols = [
            "county_id",
            "x_5070",
            "y_5070",
            "longitude",
            "latitude",
            "longitude_debug",
            "latitude_debug",
            "static_wheat_pixels",
            "static_wheat_pixels_incounty",
            "incounty_ratio",
        ]
        keep_cols = [c for c in keep_cols if c in soil.columns]
        cat1 = soil.loc[cat1_mask, keep_cols].copy()
        cat1["year"] = year
        cat1["county_id"] = county_id
        cat1["category"] = "category1_with_wheat_missing_soil"
        cat1["missing_any_soil"] = missing_any_soil.loc[cat1.index].to_numpy()
        cat1["missing_all_soil"] = missing_all_soil.loc[cat1.index].to_numpy()
        cat1["source_file"] = soil_path.name

    sat = pd.read_csv(fused_path, usecols=lambda c: c in {"longitude", "latitude", "date", "B2_count"})
    if not sat.empty and "date" in sat.columns:
        sat = sat.loc[in_10d_season_mask(sat["date"])].copy()
    sat = add_join_coords(sat) if not sat.empty else pd.DataFrame(columns=["longitude_debug", "latitude_debug"])
    sat_unique = sat.drop_duplicates(["longitude_debug", "latitude_debug"]).copy()

    if not sat_unique.empty:
        merged = sat_unique.merge(soil_keys.assign(has_soil_row=1), on=["longitude_debug", "latitude_debug"], how="left")
        sat_only = merged[merged["has_soil_row"].isna()].copy()
        if not sat_only.empty:
            # Add a useful wheat support proxy from satellite counts when available.
            if "B2_count" in sat.columns:
                sat_count = (
                    sat.groupby(["longitude_debug", "latitude_debug"], as_index=False)["B2_count"]
                    .max()
                    .rename(columns={"B2_count": "sat_max_b2_count_pixels"})
                )
                sat_only = sat_only.merge(sat_count, on=["longitude_debug", "latitude_debug"], how="left")
            else:
                sat_only["sat_max_b2_count_pixels"] = np.nan

            cat2 = sat_only[["longitude", "latitude", "longitude_debug", "latitude_debug", "sat_max_b2_count_pixels"]].copy()
            cat2["year"] = year
            cat2["county_id"] = county_id
            cat2["category"] = "category2_sat_only_no_soil_row"
            cat2["source_file"] = fused_path.name

    summary.update(
        {
            "status": "ok",
            "soil_cells": int(len(soil_keys)),
            "soil_positive_incounty_cells": int(positive_incounty.sum()),
            "category1_cells": int(len(cat1)),
            "sat_cells_10d_season": int(len(sat_unique)),
            "category2_cells": int(len(cat2)),
        }
    )
    return cat1, cat2, summary

## Run Analysis

This scans the whitelisted county-years and writes occurrence-level files. Occurrence-level means the same physical cell can appear in multiple years or counties.

In [6]:
cat1_parts = []
cat2_parts = []
summary_rows = []

for i, record in enumerate(COUNTY_YEARS, start=1):
    cat1, cat2, summary = analyze_one_county_year(record)
    if len(cat1):
        cat1_parts.append(cat1)
    if len(cat2):
        cat2_parts.append(cat2)
    summary_rows.append(summary)

    if i % 100 == 0 or i == len(COUNTY_YEARS):
        print(f"Processed {i:,}/{len(COUNTY_YEARS):,} county-years")

category1_occurrences = pd.concat(cat1_parts, ignore_index=True) if cat1_parts else pd.DataFrame()
category2_occurrences = pd.concat(cat2_parts, ignore_index=True) if cat2_parts else pd.DataFrame()
county_year_summary = pd.DataFrame(summary_rows)

category1_occurrences.to_csv(OUTPUT_CATEGORY1_OCCURRENCES, index=False)
category2_occurrences.to_csv(OUTPUT_CATEGORY2_OCCURRENCES, index=False)
county_year_summary.to_csv(OUTPUT_COUNTY_YEAR_SUMMARY, index=False)

print("Category 1 occurrences:", len(category1_occurrences))
print("Category 2 occurrences:", len(category2_occurrences))
print("Wrote:")
print(" ", OUTPUT_CATEGORY1_OCCURRENCES)
print(" ", OUTPUT_CATEGORY2_OCCURRENCES)
print(" ", OUTPUT_COUNTY_YEAR_SUMMARY)

Processed 100/1,872 county-years
Processed 200/1,872 county-years
Processed 300/1,872 county-years
Processed 400/1,872 county-years
Processed 500/1,872 county-years
Processed 600/1,872 county-years
Processed 700/1,872 county-years
Processed 800/1,872 county-years
Processed 900/1,872 county-years
Processed 1,000/1,872 county-years
Processed 1,100/1,872 county-years
Processed 1,200/1,872 county-years
Processed 1,300/1,872 county-years
Processed 1,400/1,872 county-years
Processed 1,500/1,872 county-years
Processed 1,600/1,872 county-years
Processed 1,700/1,872 county-years
Processed 1,800/1,872 county-years
Processed 1,872/1,872 county-years
Category 1 occurrences: 7183
Category 2 occurrences: 14524
Wrote:
  <repository>\missing_soil_category1_with_wheat_occurrences.csv
  <repository>\missing_soil_category2_sat_only_occurrences.csv
  <repository>\missing_soil_county_year_summary.csv


## Build Unique Download Points

This collapses repeated cells across years/county-years. If the same rounded lon/lat appears multiple times, it is listed once with aggregated metadata.

This is the file to use if soil data should be downloaded or patched only once per unique point.

In [7]:
def compact_unique_strings(values: pd.Series) -> str:
    return ";".join(map(str, sorted(pd.unique(values.dropna()))))


download_parts = []

if len(category1_occurrences):
    c1 = category1_occurrences.copy()
    c1["download_reason"] = "category1_with_wheat_missing_soil"
    c1["sat_max_b2_count_pixels"] = np.nan
    download_parts.append(c1)

if len(category2_occurrences):
    c2 = category2_occurrences.copy()
    c2["download_reason"] = "category2_sat_only_no_soil_row"
    for col in ["static_wheat_pixels", "static_wheat_pixels_incounty", "incounty_ratio", "x_5070", "y_5070"]:
        if col not in c2.columns:
            c2[col] = np.nan
    download_parts.append(c2)

all_missing = pd.concat(download_parts, ignore_index=True, sort=False) if download_parts else pd.DataFrame()

if len(all_missing):
    unique_download_points = (
        all_missing.groupby(["longitude_debug", "latitude_debug"], as_index=False)
        .agg(
            longitude=("longitude", "first"),
            latitude=("latitude", "first"),
            categories=("download_reason", compact_unique_strings),
            years=("year", compact_unique_strings),
            county_ids=("county_id", compact_unique_strings),
            n_occurrences=("year", "size"),
            n_years=("year", lambda s: s.nunique()),
            n_counties=("county_id", lambda s: s.nunique()),
            max_static_wheat_pixels=("static_wheat_pixels", "max"),
            max_static_wheat_pixels_incounty=("static_wheat_pixels_incounty", "max"),
            max_sat_b2_count_pixels=("sat_max_b2_count_pixels", "max"),
        )
        .sort_values(["categories", "longitude_debug", "latitude_debug"], kind="mergesort")
        .reset_index(drop=True)
    )
else:
    unique_download_points = pd.DataFrame()

unique_download_points.to_csv(OUTPUT_UNIQUE_DOWNLOAD_POINTS, index=False)

print("Unique download points:", len(unique_download_points))
print("Wrote:", OUTPUT_UNIQUE_DOWNLOAD_POINTS)
unique_download_points.head()

Unique download points: 15535
Wrote: <repository>\missing_soil_unique_download_points.csv


,longitude_debug,latitude_debug,longitude,latitude,categories,years,county_ids,n_occurrences,n_years,n_counties,max_static_wheat_pixels,max_static_wheat_pixels_incounty,max_sat_b2_count_pixels
0,-121.8431,39.0551,-121.843106,39.055091,category1_with_wheat_missing_soil,2025,5_51,1,1,1,137.0,43.0,NaN
1,-121.8400,39.0465,-121.839973,39.046497,category1_with_wheat_missing_soil,2025,5_51,1,1,1,89.0,66.0,NaN
2,-121.8298,38.9190,-121.829843,38.918989,category1_with_wheat_missing_soil,2025,5_51,1,1,1,271.0,136.0,NaN
3,-121.8124,38.9042,-121.812382,38.904194,category1_with_wheat_missing_soil,2025,5_51,1,1,1,328.0,10.0,NaN
4,-121.7968,38.8612,-121.796777,38.861219,category1_with_wheat_missing_soil,2025,5_51,1,1,1,461.0,1.0,NaN


## Print And Export Unique Lon/Lat Points

This gives a compact view of the exact points to patch/download. The lon/lat-only CSV is useful for external download scripts.

In [13]:
if len(unique_download_points):
    display_cols = [
        "longitude_debug",
        "latitude_debug",
        "longitude",
        "latitude",
        "categories",
        "n_occurrences",
        "n_years",
        "n_counties",
        "years",
        "county_ids",
    ]
    display_cols = [c for c in display_cols if c in unique_download_points.columns]
    points_for_print = unique_download_points[display_cols].sort_values(
        ["categories", "longitude_debug", "latitude_debug"],
        kind="mergesort",
    )

    print(f"Unique download points: {len(points_for_print):,}")
    print(points_for_print.to_string(index=False, max_rows=200))

    # CSV with just unique coordinates plus compact provenance columns.
    lonlat_cols = [
        "longitude_debug",
        "latitude_debug",
        "longitude",
        "latitude",
        "categories",
        "n_occurrences",
        "n_years",
        "n_counties",
    ]
    lonlat_cols = [c for c in lonlat_cols if c in unique_download_points.columns]
    unique_download_points[lonlat_cols].to_csv(OUTPUT_UNIQUE_DOWNLOAD_LONLAT, index=False)
    print("Wrote:", OUTPUT_UNIQUE_DOWNLOAD_LONLAT)
else:
    print("No unique download points found.")

Unique download points: 15,535
 longitude_debug  latitude_debug   longitude  latitude                        categories  n_occurrences  n_years  n_counties                         years county_ids
       -121.8431         39.0551 -121.843106 39.055091 category1_with_wheat_missing_soil              1        1           1                          2025       5_51
       -121.8400         39.0465 -121.839973 39.046497 category1_with_wheat_missing_soil              1        1           1                          2025       5_51
       -121.8298         38.9190 -121.829843 38.918989 category1_with_wheat_missing_soil              1        1           1                          2025       5_51
       -121.8124         38.9042 -121.812382 38.904194 category1_with_wheat_missing_soil              1        1           1                          2025       5_51
       -121.7968         38.8612 -121.796777 38.861219 category1_with_wheat_missing_soil              1        1           1               

## Summary Tables

Quick checks to understand scale and whether repeated cells across years are common.

In [8]:
summary = {
    "category1_occurrences": len(category1_occurrences),
    "category1_unique_points": int(category1_occurrences[["longitude_debug", "latitude_debug"]].drop_duplicates().shape[0]) if len(category1_occurrences) else 0,
    "category2_occurrences": len(category2_occurrences),
    "category2_unique_points": int(category2_occurrences[["longitude_debug", "latitude_debug"]].drop_duplicates().shape[0]) if len(category2_occurrences) else 0,
    "all_unique_download_points": len(unique_download_points),
}
summary

{'category1_occurrences': 7183,
 'category1_unique_points': 3049,
 'category2_occurrences': 14524,
 'category2_unique_points': 12522,
 'all_unique_download_points': 15535}

In [9]:
county_year_summary.sort_values(["category1_cells", "category2_cells"], ascending=False).head(30)

,year,county_id,soil_exists,fused_exists,status,soil_cells,soil_positive_incounty_cells,category1_cells,sat_cells_10d_season,category2_cells
435,2017,35_51,True,True,ok,4806,4782,43,4821,24
898,2019,35_51,True,True,ok,4852,4822,40,4590,31
160,2016,35_51,True,True,ok,4692,4665,38,4706,28
1780,2025,35_51,True,True,ok,4029,3995,34,4030,34
58,2016,17_87,True,True,ok,2147,2128,32,2154,14
1277,2022,37_6,True,True,ok,2355,2349,30,2359,8
183,2016,37_6,True,True,ok,2309,2301,30,2309,6
897,2019,35_5,True,True,ok,3968,3932,29,3978,17
915,2019,6_1,True,True,ok,2429,2396,29,2435,12
654,2018,17_87,True,True,ok,2066,2051,29,2073,11


In [10]:
if len(unique_download_points):
    display(unique_download_points["categories"].value_counts())
    display(unique_download_points.sort_values("n_occurrences", ascending=False).head(30))

categories
category2_sat_only_no_soil_row                                      12486
category1_with_wheat_missing_soil                                    3013
category1_with_wheat_missing_soil;category2_sat_only_no_soil_row       36
Name: count, dtype: int64

,longitude_debug,latitude_debug,longitude,latitude,categories,years,county_ids,n_occurrences,n_years,n_counties,max_static_wheat_pixels,max_static_wheat_pixels_incounty,max_sat_b2_count_pixels
2229,-97.6786,35.7270,-97.678633,35.727012,category1_with_wheat_missing_soil,2017;2018;2022;2023;2025,37_42;37_55,9,5,2,604.0,116.0,NaN
5858,-101.3039,36.4967,-101.303927,36.496683,category2_sat_only_no_soil_row,2016;2017;2018;2019;2020;2021;2023;2025,37_70,8,8,1,NaN,NaN,1075.0
3033,-97.9302,38.6065,-97.930211,38.606539,category1_with_wheat_missing_soil;category2_sa...,2016;2017;2018;2022;2025,17_27;17_59;17_85,8,5,3,95.0,22.0,25.0
675,-101.5307,37.7380,-101.530651,37.737966,category1_with_wheat_missing_soil,2016;2017;2022;2023;2025,17_34;17_47,7,5,2,643.0,133.0,NaN
2194,-97.7054,37.9138,-97.705363,37.913838,category1_with_wheat_missing_soil,2016;2017;2018;2025,17_40;17_87,7,4,2,323.0,71.0,NaN
5922,-101.2458,36.6333,-101.245751,36.633347,category2_sat_only_no_soil_row,2016;2017;2018;2019;2020;2021;2023,37_70,7,7,1,NaN,NaN,23.0
599,-101.8008,36.5005,-101.800820,36.500482,category1_with_wheat_missing_soil,2016;2017;2018;2019;2021;2023;2025,37_70,7,7,1,263.0,128.0,NaN
574,-101.8735,36.9977,-101.873532,36.997739,category1_with_wheat_missing_soil,2016;2017;2018;2019;2020;2021;2025,37_70,7,7,1,632.0,6.0,NaN
510,-102.0320,36.9899,-102.032009,36.989918,category1_with_wheat_missing_soil,2016;2017;2018;2019;2021;2023;2025,37_70,7,7,1,872.0,115.0,NaN
1795,-98.4840,38.5252,-98.484019,38.525240,category1_with_wheat_missing_soil,2016;2017;2018;2022;2025,17_27;17_80,7,5,2,708.0,118.0,NaN
